# ChakraNet GPU Verification (Colab Edition)
This notebook is perfectly tuned to find your files in Google Drive automatically, patch the CPU lock, and run the verification using Colab's T4 GPU.

In [1]:
# 1. Install required dependencies
!pip install -q ultralytics timm einops albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 6.3 MB/s eta 0:00:00


In [2]:
import os
import shutil
from google.colab import drive

# 2. Mount Google Drive
drive.mount('/content/drive')

# Tell Colab EXACTLY where the files are (assuming you uploaded them to the main folder of your Drive)
weight_path = '/content/drive/MyDrive/chakra_transformer_best.pth'
yolo_path = '/content/drive/MyDrive/best.pt'
zip_path = '/content/drive/MyDrive/chakramodel_data_scripts.zip'

print("\nCopying files directly (skipping the slow search)...")

if not os.path.exists(weight_path):
    print(f"\n❌ CRITICAL: Could not find the weights at {weight_path}")
else:
    print(f"\n✅ Found weights at: {weight_path}")
    os.makedirs('/content/weights', exist_ok=True)
    shutil.copy(weight_path, '/content/weights/chakra_transformer_best.pth')
    if os.path.exists(yolo_path):
        shutil.copy(yolo_path, '/content/weights/best.pt')
        print("✅ Copied YOLO weights too!")

if not os.path.exists(zip_path):
    print(f"\n❌ CRITICAL: Could not find the zip at {zip_path}")
else:
    print(f"\n✅ Found data zip at: {zip_path}")
    shutil.copy(zip_path, '/content/chakramodel_data_scripts.zip')

Mounted at /content/drive

Copying files directly (skipping the slow search)...

❌ CRITICAL: Could not find the weights at /content/drive/MyDrive/chakra_transformer_best.pth

❌ CRITICAL: Could not find the zip at /content/drive/MyDrive/chakramodel_data_scripts.zip


In [3]:
# 3. Unzip the code and dataset
!unzip -q -o /content/chakramodel_data_scripts.zip -d /content/

# CRITICAL FIX: The verify_strict.py script currently forces CPU mode.
# We need to remove the CPU mock so it uses Colab's T4 GPU!
script_path = '/content/src/verify_strict.py'
with open(script_path, 'r') as f:
    content = f.read()

# Remove the line that forces CPU
content = content.replace('torch.cuda.is_available = lambda: False', '# torch.cuda.is_available = lambda: False (Removed to use Colab GPU)')

with open(script_path, 'w') as f:
    f.write(content)
print("✅ Patched verify_strict.py to use the Colab GPU!")

unzip:  cannot find or open /content/chakramodel_data_scripts.zip, /content/chakramodel_data_scripts.zip.zip or /content/chakramodel_data_scripts.zip.ZIP.


FileNotFoundError: [Errno 2] No such file or directory: '/content/src/verify_strict.py'

In [ ]:
# 4. Run the strict verification on Colab's GPU!
!python /content/src/verify_strict.py